# 04 – Own dense TSDF: occupied / free / unknown space (Milestones 3 + 5)

Our own dense TSDF (`src/volumetric_reasoning/geometry/tsdf.py`, tests in `tests/test_tsdf.py`).
The **integration is the KinectFusion rule**; what we add is an explicit state for every voxel in the volume.

### Update per frame (for every voxel whose center projects into the image)
$z$ = depth of the voxel center in the camera, $d$ = measured depth at that pixel, $\text{sdf} = d - z$, truncation $\mu$.

| case | condition | update |
|---|---|---|
| no evidence | $d$ invalid or $d >$ `max_depth` | nothing (far / glass readings must not carve free space) |
| in front of surface | $\text{sdf} \ge \mu$ | $D \leftarrow \text{avg}(D, +1)$, $W$ += 1 |
| near surface | $-\mu < \text{sdf} < \mu$ | $D \leftarrow \text{avg}(D, \text{sdf}/\mu)$, $W$ += 1 |
| behind surface | $\text{sdf} \le -\mu$ | nothing (occluded) |

### Classification after fusion ($W \ge w_\text{min}$ = "known")
| state | rule |
|---|---|
| **unknown** | $W < w_\text{min}$: never / rarely seen, or only ever hidden behind surfaces |
| **occupied** | surface passes through the voxel: $\lvert D\mu \rvert \le$ half voxel diagonal |
| **free** | $D\mu >$ half voxel diagonal: in front of every observed surface |
| **behind surface** | $D\mu <$ − half voxel diagonal: just behind a surface (inside the truncation band) |

Decisions taken for now (all configurable): constant weights (as KinectFusion), depth beyond `max_depth` ignored,
occupied = surface voxels. Whether *behind surface* counts as occupied is open.

**Coarse voxels.** Fusing directly at 20–50 cm would need truncation distances of 1–2 m and destroy the geometry.
Instead we fuse at 2 cm and **pool** blocks of fine voxels (`pool_states`): occupied if any fine voxel is occupied/behind,
else unknown if > 50 % unknown, else free. This is one possible rule, not a settled choice.

In [ ]:
%matplotlib inline
import time
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch
import open3d as o3d

from volumetric_reasoning.config import load_config, resolve_path
from volumetric_reasoning.datasets import ScanNetPPIphoneScene
from volumetric_reasoning.geometry.tsdf import (DenseTSDF, bounds_from_points, count_states, pool_states,
                                                UNKNOWN, FREE, OCCUPIED, BEHIND, STATE_NAMES)
from volumetric_reasoning.geometry.tsdf_open3d import integrate_open3d, extract_mesh as extract_mesh_o3d
from volumetric_reasoning.geometry.backprojection import frames_to_points
from volumetric_reasoning.geometry.mesh_eval import compare_meshes
from volumetric_reasoning.geometry.raycast import make_raycasting_scene, render_topdown

# ---- settings -----------------------------------------------------------------
SCENE_ID = "1ada7a0617"   # None = use dataset.scene_id from the config
N_FRAMES = None           # None = every tsdf.frame_stride-th frame (~276); e.g. 32 = 32 frames spread over the video
VOXEL_SIZE = 0.02         # fine fusion resolution (m)
W_MIN = 3                 # min. observations for a voxel to count as known (use 1 for <= ~64 frames, see section 5)
# ----------------------------------------------------------------------------------

cfg = load_config()
if SCENE_ID is not None:
    cfg["dataset"]["scene_id"] = SCENE_ID
tc, dc = cfg["tsdf"], cfg["depth"]
scene = ScanNetPPIphoneScene(resolve_path(cfg["dataset"]["root"]), cfg["dataset"]["scene_id"])
indices = scene.select_uniform(N_FRAMES) if N_FRAMES else scene.select_frames(stride=tc["frame_stride"])
frames = scene.load_frames(indices, with_rgb=False)
ref_mesh = o3d.io.read_triangle_mesh(str(scene.mesh_path))

# Grid bounds from our own back-projected points (robust percentiles), not from the laser mesh.
pts, _, _ = frames_to_points(frames[:: max(1, len(frames) // 70)], dc["min_depth"], dc["max_depth"])
lo, hi = bounds_from_points(pts, margin=0.1)
print(f"{scene.scene_id}: {len(frames)} frames, grid bounds {lo.round(2)} .. {hi.round(2)} m")

STATE_COLORS = ListedColormap(["#d9d9d9", "#9ecae1", "#d62728", "#fdae6b"])  # unknown, free, occupied, behind
LEGEND = [Patch(color=STATE_COLORS(i), label=STATE_NAMES[i]) for i in range(4)]

## 1. Fuse

In [ ]:
grid = DenseTSDF.from_bounds(lo, hi, VOXEL_SIZE, tc["trunc_voxels"])
t0 = time.time()
grid.integrate_frames(frames, dc["min_depth"], dc["max_depth"])
print(f"grid {grid.dims} = {grid.num_voxels:,} voxels, integration {time.time() - t0:.1f} s")
state = grid.classify(W_MIN)
for name, n in count_states(state).items():
    print(f"  {name:15s} {n:>11,}  ({n / grid.num_voxels * 100:5.1f} %)")
print("Note: the grid is an axis-aligned box around a slightly rotated room, so part of 'unknown' lies outside the walls.")

## 2. Validation: our mesh vs. Open3D vs. laser scan

Marching Cubes on our TSDF (triangles touching voxels with $W < w_\text{min}$ are dropped).

In [ ]:
mesh = grid.extract_mesh(W_MIN)
mesh_o3d = extract_mesh_o3d(integrate_open3d(frames, VOXEL_SIZE, tc["trunc_voxels"], dc["min_depth"], dc["max_depth"], with_color=False), W_MIN)
for name, m in [("ours", mesh), ("Open3D", mesh_o3d)]:
    r = compare_meshes(m, ref_mesh)
    print(f"{name:7s} triangles {len(m.triangles):>9,}  acc {r['accuracy_median_m']*100:.1f} cm  comp {r['completeness_median_m']*100:.1f} cm  "
          f"P {r['precision']:.3f}  R {r['recall']:.3f}  F {r['fscore']:.3f}")

Z_CUT = 2.0
xy_min, xy_max = lo[:2], hi[:2]
ext = [xy_min[0], xy_max[0], xy_min[1], xy_max[1]]
fig, ax = plt.subplots(1, 3, figsize=(18, 6))
for a, m, title in zip(ax, [mesh, mesh_o3d, ref_mesh], ["ours", "Open3D", "laser mesh"]):
    a.imshow(render_topdown(make_raycasting_scene(m), xy_min, xy_max, Z_CUT, 0.01), cmap="gray", vmin=0, vmax=1, extent=ext)
    a.set_title(f"{title} (top view, cut at {Z_CUT} m)")
plt.tight_layout()

## 3. Horizontal slices through the volume

Each image is one layer of voxels at the given height. Expected: walls and furniture **occupied** (red), the room
**free** (blue), a thin **behind-surface** band (orange) behind walls/objects, and **unknown** (gray) outside the walls,
under desks and inside furniture.

In [ ]:
HEIGHTS = [0.1, 0.5, 0.9, 1.4, 2.0, 2.6]
zs = grid.origin[2] + (np.arange(grid.dims[2]) + 0.5) * grid.voxel_size
fig, ax = plt.subplots(2, 3, figsize=(18, 11))
for a, h in zip(ax.ravel(), HEIGHTS):
    k = int(np.argmin(np.abs(zs - h)))
    a.imshow(state[:, :, k].T, origin="lower", cmap=STATE_COLORS, vmin=0, vmax=3, extent=ext, interpolation="nearest")
    a.set_title(f"z = {zs[k]:.2f} m")
ax[0, 0].legend(handles=LEGEND, loc="upper left", fontsize=9)
plt.tight_layout()

### How often was each voxel observed?

Weight $W$ (number of observations) in one slice, and the per-height fraction of each state.

In [ ]:
k = int(np.argmin(np.abs(zs - 1.0)))
fig, ax = plt.subplots(1, 2, figsize=(16, 5.5))
im = ax[0].imshow(grid.weight[:, :, k].T, origin="lower", extent=ext, cmap="magma")
plt.colorbar(im, ax=ax[0], label="observations W"); ax[0].set_title(f"weight at z = {zs[k]:.2f} m")
frac = np.stack([(state == s).mean(axis=(0, 1)) for s in range(4)])
ax[1].stackplot(zs, frac * 100, colors=[STATE_COLORS(i) for i in range(4)], labels=[STATE_NAMES[i] for i in range(4)])
ax[1].set_xlabel("height z [m]"); ax[1].set_ylabel("% of voxels in layer"); ax[1].legend(loc="lower right"); ax[1].set_title("state per height")
plt.tight_layout()

## 4. Coarser voxels (pooled from the 2 cm grid)

Token counts in context: a VLM typically sees e.g. 32 frames × 64–256 visual tokens ≈ 2k–8k tokens.

In [ ]:
FACTORS = [1, 2, 4, 8, 16, 32]   # pooled size = factor * VOXEL_SIZE
rows, pooled = [], {}
for fct in FACTORS:
    p = pool_states(state, fct)
    pooled[fct] = p
    rows.append({"size_cm": fct * VOXEL_SIZE * 100, **count_states(p), "total": p.size})
print(f"{'voxel':>7} {'occupied':>10} {'free':>10} {'unknown':>10} {'total':>11}")
for r in rows:
    print(f"{r['size_cm']:5.0f}cm {r['occupied']:>10,} {r['free']:>10,} {r['unknown']:>10,} {r['total']:>11,}")

fig, ax = plt.subplots(1, 2, figsize=(14, 4.2))
x = np.arange(len(rows))
for s, off in [(OCCUPIED, -0.25), (FREE, 0), (UNKNOWN, 0.25)]:
    ax[0].bar(x + off, [r[STATE_NAMES[s]] for r in rows], width=0.25, color=STATE_COLORS(s), label=STATE_NAMES[s])
ax[0].set_yscale("log"); ax[0].set_xticks(x, [f"{r['size_cm']:.0f} cm" for r in rows]); ax[0].set_ylabel("voxels")
ax[0].axhspan(2000, 8000, color="green", alpha=0.1, label="typical VLM token budget"); ax[0].legend()
ax[0].set_title("voxels per state vs. voxel size")
ax[1].plot([r["size_cm"] for r in rows], [r["occupied"] / (r["occupied"] + r["free"]) * 100 for r in rows], "o-")
ax[1].set_xscale("log"); ax[1].set_xlabel("voxel size [cm]"); ax[1].set_ylabel("% occupied among known voxels")
ax[1].set_title("occupied share grows with voxel size (any-occupied rule)")
plt.tight_layout()

h = 1.0
fig, ax = plt.subplots(1, len(FACTORS), figsize=(4 * len(FACTORS), 4))
for a, fct in zip(ax, FACTORS):
    p = pooled[fct]
    kz = min(int((h - grid.origin[2]) / (fct * VOXEL_SIZE)), p.shape[2] - 1)
    a.imshow(p[:, :, kz].T, origin="lower", cmap=STATE_COLORS, vmin=0, vmax=3, interpolation="nearest")
    a.set_title(f"{fct * VOXEL_SIZE * 100:.0f} cm, layer at ~{h} m"); a.axis("off")
plt.tight_layout()

## 4b. Marching Cubes meshes at different voxel sizes

The pooled grids above only store states, not distances, so they cannot be meshed. Here we **fuse directly** at each
voxel size (truncation = `trunc_voxels` × voxel size, i.e. 16 cm at 4 cm voxels, 1.28 m at 32 cm) and extract the mesh.
This shows how much geometry a grid of that resolution can represent at all. Meshes are saved to `outputs/<scene>/`.

In [ ]:
MESH_SIZES = [0.02, 0.04, 0.08, 0.16, 0.32, 0.64]
out_dir = resolve_path(cfg["output"]["root"]) / scene.scene_id
out_dir.mkdir(parents=True, exist_ok=True)
meshes, mesh_rows = {}, []
for vs in MESH_SIZES:
    if np.isclose(vs, VOXEL_SIZE):
        g, secs = grid, float("nan")  # reuse the grid fused in section 1
    else:
        g = DenseTSDF.from_bounds(lo, hi, vs, tc["trunc_voxels"])
        t0 = time.time(); g.integrate_frames(frames, dc["min_depth"], dc["max_depth"]); secs = time.time() - t0
    m = g.extract_mesh(W_MIN)
    meshes[vs] = m
    o3d.io.write_triangle_mesh(str(out_dir / f"tsdf_ours_{int(round(vs * 1000))}mm.ply"), m)
    r = compare_meshes(m, ref_mesh, n_samples=100_000)
    mesh_rows.append({"voxel_cm": vs * 100, "trunc_cm": g.trunc * 100, "seconds": secs, "triangles": len(m.triangles), **r})

print(f"{'voxel':>6} {'trunc':>6} {'time':>6} {'triangles':>10} {'acc':>6} {'comp':>6} {'P':>6} {'R':>6} {'F@5cm':>6}")
for r in mesh_rows:
    print(f"{r['voxel_cm']:4.0f}cm {r['trunc_cm']:4.0f}cm {r['seconds']:5.1f}s {r['triangles']:>10,} {r['accuracy_median_m']*100:5.1f}c "
          f"{r['completeness_median_m']*100:5.1f}c {r['precision']:6.3f} {r['recall']:6.3f} {r['fscore']:6.3f}")

plt.figure(figsize=(6, 3.5))
plt.semilogx([r["voxel_cm"] for r in mesh_rows], [r["fscore"] for r in mesh_rows], "o-", label="F-score @ 5 cm")
plt.semilogx([r["voxel_cm"] for r in mesh_rows], [r["precision"] for r in mesh_rows], "s--", label="precision")
plt.semilogx([r["voxel_cm"] for r in mesh_rows], [r["recall"] for r in mesh_rows], "^--", label="recall")
plt.xlabel("voxel size [cm]"); plt.legend(); plt.title("mesh quality vs. voxel size")

# Top view (ceiling cut) per voxel size, plus the laser mesh for reference.
fig, ax = plt.subplots(1, len(MESH_SIZES) + 1, figsize=(4 * (len(MESH_SIZES) + 1), 4))
for a, vs in zip(ax, MESH_SIZES):
    a.imshow(render_topdown(make_raycasting_scene(meshes[vs]), xy_min, xy_max, Z_CUT, 0.01), cmap="gray", vmin=0, vmax=1, extent=ext)
    a.set_title(f"{vs * 100:.0f} cm voxels"); a.axis("off")
ax[-1].imshow(render_topdown(make_raycasting_scene(ref_mesh), xy_min, xy_max, Z_CUT, 0.01), cmap="gray", vmin=0, vmax=1, extent=ext)
ax[-1].set_title("laser mesh"); ax[-1].axis("off")
plt.tight_layout()

# Same camera view for every voxel size.
from volumetric_reasoning.geometry.raycast import render_normals
f = frames[len(frames) // 2]
W_R, H_R = 512, 384
K_R = f.K_depth * np.array([[W_R / f.depth.shape[1]], [H_R / f.depth.shape[0]], [1]])
fig, ax = plt.subplots(1, len(MESH_SIZES) + 1, figsize=(4 * (len(MESH_SIZES) + 1), 3.3))
for a, vs in zip(ax, MESH_SIZES):
    a.imshow(render_normals(make_raycasting_scene(meshes[vs]), K_R, f.cam_to_world, W_R, H_R), cmap="gray", vmin=0, vmax=1)
    a.set_title(f"{vs * 100:.0f} cm voxels"); a.axis("off")
ax[-1].imshow(render_normals(make_raycasting_scene(ref_mesh), K_R, f.cam_to_world, W_R, H_R), cmap="gray", vmin=0, vmax=1)
ax[-1].set_title(f"laser mesh (frame {f.index})"); ax[-1].axis("off")
plt.tight_layout()

### 3D view of the meshes (and saving them)

`SIZES_TO_SHOW` lists the voxel sizes to display; they are placed **side by side** (shifted along x), with the laser mesh
at the end if `SHOW_LASER = True`. `MAX_HEIGHT` removes the ceiling (`None` = full mesh).

* `SHOW_3D = True` opens an Open3D window (left-drag rotates, scroll zooms, right-drag pans; press `P` to save a
  screenshot PNG into the notebook's folder).
* `SAVE = True` writes the whole arrangement as **one PLY file** to `outputs/<scene>/` (opens in MeshLab, Blender or
  Open3D, e.g. to send to someone). The laser mesh is large (~1.5 M triangles); `LASER_TRIANGLES` simplifies it for the file.
  Note: the laser mesh is ScanNet++ data, so only share it with people who have dataset access.

In [ ]:
SHOW_3D = False
SAVE = False
SIZES_TO_SHOW = [0.02, 0.08, 0.16, 0.32]   # any of MESH_SIZES; e.g. [0.08] shows just that mesh
SHOW_LASER = True
MAX_HEIGHT = 2.0
GAP = 0.5                                   # meters between neighboring meshes
LASER_TRIANGLES = 300_000                   # simplify the laser mesh for the saved file (None = keep all)

if SHOW_3D or SAVE:
    to_show = [(f"{vs * 100:.0f}cm", meshes[vs]) for vs in SIZES_TO_SHOW]
    if SHOW_LASER:
        to_show.append(("laser", ref_mesh))
    geoms, x_offset = [], 0.0
    for name, m in to_show:
        m = o3d.geometry.TriangleMesh(m)  # copy, so the originals stay untouched
        if MAX_HEIGHT is not None:
            lo_b, hi_b = m.get_min_bound(), m.get_max_bound()
            m = m.crop(o3d.geometry.AxisAlignedBoundingBox(lo_b, [hi_b[0], hi_b[1], MAX_HEIGHT]))
        m.translate([x_offset - m.get_min_bound()[0], 0, 0])
        if not m.has_vertex_colors():
            m.paint_uniform_color([0.75, 0.75, 0.75])
        m.compute_vertex_normals()
        geoms.append(m)
        print(f"{name:>6}: x from {x_offset:.1f} m, {len(m.triangles):,} triangles")
        x_offset = m.get_max_bound()[0] + GAP

    if SAVE:
        combined = o3d.geometry.TriangleMesh()
        for (name, _), g in zip(to_show, geoms):
            if name == "laser" and LASER_TRIANGLES and len(g.triangles) > LASER_TRIANGLES:
                g = g.simplify_quadric_decimation(LASER_TRIANGLES)
            combined += g
        tag = "_".join(name for name, _ in to_show)
        path = out_dir / f"mesh_comparison_{tag}.ply"
        o3d.io.write_triangle_mesh(str(path), combined, write_ascii=False, compressed=True)
        print(f"saved {len(combined.triangles):,} triangles -> {path} ({path.stat().st_size / 1e6:.1f} MB)")

    if SHOW_3D:
        center = np.mean([g.get_center() for g in geoms], axis=0)
        o3d.visualization.draw_geometries(geoms, front=[0.0, -0.5, 1.0], up=[0, 0, 1], lookat=center, zoom=0.45)

## 5. How many frames are needed?

Frames spread uniformly over the video. With few frames most voxels are observed only once or twice, so
$w_\text{min} = 3$ discards most of the scene; $w_\text{min}$ has to scale with the number of frames.
Runtime is roughly 0.3 s per frame at 2 cm.

In [ ]:
FRAME_COUNTS = [64, 32, 16, 8]
results = []
for n in FRAME_COUNTS:
    fr = scene.load_frames(scene.select_uniform(n), with_rgb=False)
    g = DenseTSDF.from_bounds(lo, hi, VOXEL_SIZE, tc["trunc_voxels"])
    g.integrate_frames(fr, dc["min_depth"], dc["max_depth"])
    for w in [1, 3]:
        r = compare_meshes(g.extract_mesh(w), ref_mesh, n_samples=100_000)
        c = count_states(g.classify(w))
        results.append({"frames": n, "w_min": w, "fscore": r["fscore"], "recall": r["recall"], "precision": r["precision"],
                        **{k: v / g.num_voxels * 100 for k, v in c.items()}})
    print(f"{n:3d} frames done")

fig, ax = plt.subplots(1, 3, figsize=(17, 4))
for w, ls in [(1, "o-"), (3, "s--")]:
    rs = [r for r in results if r["w_min"] == w]
    ax[0].plot([r["frames"] for r in rs], [r["fscore"] for r in rs], ls, label=f"w_min={w}")
    ax[1].plot([r["frames"] for r in rs], [r["unknown"] for r in rs], ls, label=f"w_min={w}")
    ax[2].plot([r["frames"] for r in rs], [r["free"] for r in rs], ls, label=f"w_min={w}")
for a, t in zip(ax, ["mesh F-score @ 5 cm", "% unknown voxels", "% free voxels"]):
    a.set_xscale("log", base=2); a.set_xlabel("frames"); a.set_title(t); a.legend()
plt.tight_layout()

## 6. Optional: 3D view of coarse voxels

Occupied voxels as cubes (colored by height), free voxels as small blue points. `MAX_HEIGHT` removes the ceiling.
(For the Marching Cubes meshes see the 3D cell in section 4b.)

In [ ]:
SHOW_3D = False
FACTOR = 4          # 4 * 2 cm = 8 cm voxels
MAX_HEIGHT = 2.0
if SHOW_3D:
    p = pooled[FACTOR]; size = FACTOR * VOXEL_SIZE
    centers = lambda s: grid.origin + (np.argwhere(p == s) + 0.5) * size
    occ, free = centers(OCCUPIED), centers(FREE)
    occ, free = occ[occ[:, 2] < MAX_HEIGHT], free[free[:, 2] < MAX_HEIGHT]
    pc = o3d.geometry.PointCloud(o3d.utility.Vector3dVector(occ))
    pc.colors = o3d.utility.Vector3dVector(plt.cm.viridis((occ[:, 2] - occ[:, 2].min()) / np.ptp(occ[:, 2]))[:, :3])
    vg = o3d.geometry.VoxelGrid.create_from_point_cloud(pc, voxel_size=size)
    fp = o3d.geometry.PointCloud(o3d.utility.Vector3dVector(free)); fp.paint_uniform_color([0.4, 0.6, 1.0])
    o3d.visualization.draw_geometries([vg, fp], front=[0.0, -0.4, 1.0], up=[0, 0, 1], lookat=occ.mean(0), zoom=0.6)

## Findings

- Fill in after running.
- Occupied share among known voxels grows from ~5 % (2 cm) to ~56 % (32 cm) with the any-occupied rule,
  although the room is mostly empty: coarse voxels touching any surface count as occupied.
- Open questions: pooling rule for coarse voxels (any-occupied vs. majority vs. storing fractions), `behind surface` = occupied or unknown?,
  w_min as a function of frame count, glass (scene 3e8bba0176) creating false free space.